# Week 2: The Renewal Cliff — Predicting Contract Churn 90 Days Out
## Cox Proportional Hazards Model

**Masterclass:** [Week 2 Deep-Dive](https://balaviswanath-analytics.github.io/analytics_masterclass/week2_renewal_cliff.html)

**Dataset:** IBM Telco Customer Churn (Kaggle)
- Download: https://www.kaggle.com/datasets/blastchar/telco-customer-churn
- Save as `WA_Fn-UseC_-Telco-Customer-Churn.csv` in this directory

**What You'll Build:**
1. Survival data preparation from a real telecom dataset
2. KM curves by contract type
3. Cox PH model with hazard ratio interpretation
4. Proportional hazards assumption testing
5. Individual survival predictions and risk scoring

In [ ]:
!pip install lifelines pandas matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lifelines import CoxPHFitter, KaplanMeierFitter
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print("Libraries loaded.")

---
## Part 1: Load and Prepare the Telco Dataset

In [ ]:
try:
    df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
except FileNotFoundError:
    print("Dataset not found!")
    print("Download from: https://www.kaggle.com/datasets/blastchar/telco-customer-churn")
    print("Save as 'WA_Fn-UseC_-Telco-Customer-Churn.csv' in this directory.")
    raise

# Clean
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna(subset=['TotalCharges'])

# Binary encoding for survival model
df['churned'] = (df['Churn'] == 'Yes').astype(int)
df['is_female'] = (df['gender'] == 'Female').astype(int)
df['has_partner'] = (df['Partner'] == 'Yes').astype(int)
df['has_dependents'] = (df['Dependents'] == 'Yes').astype(int)
df['paperless'] = (df['PaperlessBilling'] == 'Yes').astype(int)
df['contract_one_year'] = (df['Contract'] == 'One year').astype(int)
df['contract_two_year'] = (df['Contract'] == 'Two year').astype(int)
df['fiber_optic'] = (df['InternetService'] == 'Fiber optic').astype(int)
df['no_internet'] = (df['InternetService'] == 'No').astype(int)
df['electronic_check'] = (df['PaymentMethod'] == 'Electronic check').astype(int)

print(f"Customers: {len(df):,} | Churned: {df['churned'].sum():,} ({df['churned'].mean():.1%})")
print(f"Median tenure: {df['tenure'].median():.0f} months")

---
## Part 2: KM Curves by Contract Type

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = {'Month-to-month': '#c45d3e', 'One year': '#e9c46a', 'Two year': '#2d6a4f'}

for ctype in ['Month-to-month', 'One year', 'Two year']:
    mask = df['Contract'] == ctype
    kmf = KaplanMeierFitter()
    kmf.fit(df.loc[mask, 'tenure'], df.loc[mask, 'churned'], label=ctype)
    kmf.plot_survival_function(ax=ax, color=colors[ctype], linewidth=2)

ax.set_title('Customer Survival by Contract Type', fontsize=14, fontweight='bold')
ax.set_xlabel('Tenure (Months)')
ax.set_ylabel('Survival Probability')
ax.legend(title='Contract Type')
plt.tight_layout()
plt.show()

# Quick stats
for ctype in ['Month-to-month', 'One year', 'Two year']:
    rate = df.loc[df['Contract']==ctype, 'churned'].mean()
    print(f"  {ctype}: {rate:.1%} churn rate")

---
## Part 3: Cox Proportional Hazards Model

Cox PH answers: **"Which factors increase or decrease the hazard (instantaneous churn risk)?"** while controlling for all other variables.

The key output is the **Hazard Ratio (HR):**
- HR > 1 → increases churn risk
- HR < 1 → decreases churn risk (protective)
- HR = 1 → no effect

In [ ]:
covariates = [
    'is_female', 'SeniorCitizen', 'has_partner', 'has_dependents',
    'contract_one_year', 'contract_two_year',
    'fiber_optic', 'no_internet',
    'paperless', 'electronic_check', 'MonthlyCharges'
]

cph = CoxPHFitter(penalizer=0.01)
cph.fit(df[covariates + ['tenure', 'churned']],
        duration_col='tenure', event_col='churned')
cph.print_summary()

In [ ]:
# Forest plot of hazard ratios
fig, ax = plt.subplots(figsize=(10, 8))
cph.plot(ax=ax)
ax.set_title('Cox PH: Hazard Ratios', fontsize=14, fontweight='bold')
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

# Plain English interpretation
hrs = np.exp(cph.params_)
print("\n=== Top Risk Factors ===")
for var in hrs.nlargest(3).index:
    print(f"  {var}: HR={hrs[var]:.2f} → {(hrs[var]-1)*100:.0f}% higher churn rate")
print("\n=== Top Protective Factors ===")
for var in hrs.nsmallest(3).index:
    print(f"  {var}: HR={hrs[var]:.2f} → {(1-hrs[var])*100:.0f}% lower churn rate")

---
## Part 4: Test the Proportional Hazards Assumption

**Critical step.** If violated, HRs are unreliable. The Schoenfeld residual test checks whether each covariate's effect is constant over time.

In [ ]:
results = cph.check_assumptions(df[covariates + ['tenure', 'churned']],
                                p_value_threshold=0.05, show_plots=False)

---
## Part 5: Individual Risk Scoring & Survival Predictions

In [ ]:
# Compare two customer profiles
high_risk = pd.DataFrame({
    'is_female': [0], 'SeniorCitizen': [1], 'has_partner': [0],
    'has_dependents': [0], 'contract_one_year': [0], 'contract_two_year': [0],
    'fiber_optic': [1], 'no_internet': [0], 'paperless': [1],
    'electronic_check': [1], 'MonthlyCharges': [95]
})
low_risk = pd.DataFrame({
    'is_female': [1], 'SeniorCitizen': [0], 'has_partner': [1],
    'has_dependents': [1], 'contract_one_year': [0], 'contract_two_year': [1],
    'fiber_optic': [0], 'no_internet': [0], 'paperless': [0],
    'electronic_check': [0], 'MonthlyCharges': [45]
})

fig, ax = plt.subplots(figsize=(10, 6))
cph.predict_survival_function(high_risk).plot(ax=ax, color='#c45d3e', linewidth=2, label='High Risk')
cph.predict_survival_function(low_risk).plot(ax=ax, color='#2d6a4f', linewidth=2, label='Low Risk')
ax.set_title('Predicted Survival: High vs Low Risk', fontsize=14, fontweight='bold')
ax.set_xlabel('Months')
ax.set_ylabel('Survival Probability')
ax.legend()
plt.tight_layout()
plt.show()

print("High Risk: Senior, month-to-month, fiber, e-check, $95/mo")
print("Low Risk: Non-senior, 2yr contract, DSL, bank transfer, $45/mo")

In [ ]:
# Score entire active customer base
df['risk_score'] = cph.predict_partial_hazard(df[covariates]).values
active = df[df['churned']==0].copy()
active['risk_pct'] = active['risk_score'].rank(pct=True)

# Top 100 at-risk active customers
top100 = active.nlargest(100, 'risk_score')
print("=== Top 100 At-Risk Active Customers ===")
print(f"Avg monthly charges: ${top100['MonthlyCharges'].mean():.0f}")
print(f"Avg tenure: {top100['tenure'].mean():.0f} months")
print(f"Monthly revenue at stake: ${top100['MonthlyCharges'].sum():,.0f}")
print(f"\nContract breakdown:")
print(top100['Contract'].value_counts().to_string())

---
## Exercises

### Exercise 1: Add Service Covariates
Add `OnlineSecurity`, `TechSupport`, `StreamingTV`, `StreamingMovies` as binary covariates. Which service is most protective against churn?

### Exercise 2: Stratified Cox
Fit a stratified model using `Contract` as strata (separate baseline hazards, shared covariate effects). Compare HRs to the unstratified model.

```python
cph_strat = CoxPHFitter(penalizer=0.01)
cph_strat.fit(df[covs + ['tenure', 'churned', 'Contract']],
              duration_col='tenure', event_col='churned',
              strata=['Contract'])
```

### Exercise 3: 90-Day Intervention List
Identify all active customers with predicted 6-month survival < 50%. How many customers? What's total monthly revenue at risk?

---
## Key Takeaways

1. **Cox PH** reveals *why* customers churn (not just when)
2. **Hazard ratios** quantify relative impact while controlling for confounders
3. **Always test PH assumption** — violations make HRs unreliable
4. **Risk scoring** converts models into actionable intervention lists

**Next:** [Week 3 — Churn Segmentation](https://balaviswanath-analytics.github.io/analytics_masterclass/week3_churn_segmentation.html)